### [한번 해보기] 음식 리뷰 데이터 활용 유사도 검색

- corpus -> embedding vector -> 유사도 기반 검색

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

In [5]:
import pandas as pd
import tqdm
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Summary : title
# Text : content
def text_to_embedding(texts, model="text-embedding-3-small"):
    texts = [text.replace("\n", " ") for text in texts]
    response = client.embeddings.create(
        model=model,
        input=texts
    )
    return [data.embedding for data in response.data]

# Content -> Embedding Vector1 (csv 저장)
# 사용자 입력 (best coffee) -> Embedding Vector2
# Vector1, Vector2 -> 유사도 계산 -> 유사한 리뷰 반환

def search_similar_reviews(query, review_embeddings, texts_df, top_k=5):
    query_embedding = np.array(text_to_embedding([query])[0]).reshape(1, -1)
    sims = cosine_similarity(query_embedding, review_embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]

    result = texts_df.iloc[top_idx].copy()
    result["similarity"] = sims[top_idx]
    return result[["Summary", "Text", "similarity"]]

# 데이터 로드
texts_df = pd.read_csv('data/fine_food_reviews_1k.csv')
texts_df = texts_df[['Summary', 'Text']].fillna("")

# 리뷰 텍스트 합치기
texts_df["combined"] = (
    "Summary: " + texts_df["Summary"].astype(str) +
    " | Text: " + texts_df["Text"].astype(str)
)

documents = texts_df["combined"].tolist()
all_embeddings = np.array(text_to_embedding(documents))

# 검색
query = "best coffee"
result = search_similar_reviews(query, all_embeddings, texts_df, top_k=5)

# 출력
for i, row in result.iterrows():
    print(f"[유사도] {row['similarity']:.4f}")
    print(f"[Summary] {row['Summary']}")
    print(f"[Text] {row['Text']}")
    print("-" * 80)

[유사도] 0.6046
[Summary] Great Coffee
[Text] I have a coffee maker that grinds my coffee beans.  It's hard to find whole bean decafinated coffee.  When I find it in the brand that I like, I am excited.  Seattle's Best is my favorite.
--------------------------------------------------------------------------------
[유사도] 0.5941
[Summary] Better than you-know-who's coffee...
[Text] So my wife is a latte freak, and nursing, so decaf is the approved type.  After the Senseo left the market, I struggled and found the <a href="http://www.amazon.com/gp/product/B0047BIWSK">Aerobie AeroPress Coffee and Espresso Maker</a> which is like a French Press for the 21st century.  After getting our recipe figured out, my wife, who's been buying Venti Decaf Latte's at $4 a pop almost daily for years now declares that Seattle's best Level 3 Decaf in her home-made Latte is the best coffee she can get.  We've tried other bands, and this is her favorite, hands down!
----------------------------------------------